# 04. Final Coloring Book Pipeline

이 노트북은 01, 02, 03번 노트북에서 검증한 단계를 하나로 연결합니다.

입력:

- `data/flowers.jpg` 입력 이미지
- 원하는 색상 개수 K
- 경계선 Canny 임계값
- 선 두께
- 최소 영역 크기

출력:

1. 원본 이미지
2. K-Means, Posterization, Median Cut 색상 단순화 비교
3. 색상 오차와 실제 고유색 수 비교
4. Sobel, Laplacian, Canny, Hybrid Color Boundary 경계선 비교
5. Connected Components 영역 분리 결과
6. 색상 번호가 들어간 최종 컬러링북 이미지
7. RGB 색상표와 색상 인덱스 표
8. K 변화에 따른 복잡도 비교

In [ ]:
# Kernel -> Restart & Run All ? ???? ?????.
import os
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# ???? project ??/server ??? ???? server/data? server/outputs? ?????.
CURRENT_DIR = Path.cwd()
FALLBACK_SERVER_DIR = Path(r"C:\Users\qhtm0\Desktop\project\HCI_PROJECT\server")
if (CURRENT_DIR / "data").is_dir() and (CURRENT_DIR / "src").is_dir():
    SERVER_DIR = CURRENT_DIR
elif (CURRENT_DIR / "server" / "data").is_dir() and (CURRENT_DIR / "server" / "src").is_dir():
    SERVER_DIR = CURRENT_DIR / "server"
else:
    SERVER_DIR = FALLBACK_SERVER_DIR

SRC_DIR = SERVER_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

import importlib
import coloringbook_utils as coloringbook_utils
importlib.reload(coloringbook_utils)
from coloringbook_utils import *

# coloringbook_utils? DATA_DIR/OUTPUT_DIR ??? import? ?, ???? ????? ?? ?????.
DATA_DIR = SERVER_DIR / "data"
OUTPUT_DIR = SERVER_DIR / "outputs"

ensure_dirs()
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 120

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGE_PATHS = sorted(
    str(path)
    for path in DATA_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not IMAGE_PATHS:
    raise FileNotFoundError(f"No input images found in {DATA_DIR}")

print(f"Server dir: {SERVER_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Input images: {len(IMAGE_PATHS)}")
for image_path in IMAGE_PATHS:
    print(f"- {image_path}")


## 전체 파라미터

실험 발표에서는 `K=5, 10, 20`을 비교하고, 최종 결과는 보통 `K=10`부터 조정하는 방식이 좋습니다.

In [ ]:
# ? ???? ?? ?? ?? ?? ?? K + 10?? ?? ?????.
MIN_K_CANDIDATES = list(range(2, 31))
RETENTION_THRESHOLD = 0.75
K_MARGIN = 10
K_ESTIMATION_MAX_SIZE = 420

# Canny ??????. ?? ??? ???? ??? ??? ??? ??? ? ????.
CANNY_LOW = 80
CANNY_HIGH = 180

# ?? Hybrid edge? K-Means ?? ?? ??? ??? Lab ? ??? ? ??? ?????.
COLOR_EDGE_DELTA = 35
FINAL_CLOSE_ITER = 4

# ?? ?? edge ?? ?????. ?? ? ??? ??? ?? ?? ????.
OBJECT_MIN_AREA = 260
OBJECT_CLOSE_KERNEL = 5

# ???, ?, ?? ?? ?? ???? ?? ???? ?? ?? ?? ???? ????.
DETAIL_CANNY_LOW = 90
DETAIL_CANNY_HIGH = 200
DETAIL_MIN_AREA = 3
DETAIL_MAX_AREA = 1200
DARK_DETAIL_THRESHOLD = 70
DARK_DETAIL_MIN_AREA = 5
DARK_DETAIL_MAX_AREA = 900
DETAIL_SUPPRESS_RADIUS = 2

# ?? ??/??/???? ? ??? ??? ???? ?? ??? ?????.
INK_LINE_THRESHOLD = 135
INK_NEUTRAL_DELTA = 85
INK_MIN_AREA = 8
INK_MASK_DILATE = 1

# ?? ???? ? ?????. ?? ?? ?? ?????.
LINE_THICKNESS = 1
EDGE_COMPARE_THICKNESS = 1
MIN_REGION_AREA = 160

# ???, ??, ??????? ?? ? ??? ??? ?? ?? ??? ?????.
SIMPLIFY_DIAMETER = 9
SIMPLIFY_SIGMA_COLOR = 90
SIMPLIFY_SIGMA_SPACE = 90

# ???/?? ??? ?? ?? ? ??? ?? ????? ?????.
SHADOW_MERGE_ENABLED = True
SHADOW_MERGE_MAX_AREA = 450
SHADOW_MERGE_COLOR_DELTA = 45
SHADOW_MERGE_PASSES = 2

def lab_color_spread_for_k(image):
    """Return the original image's Lab color spread for minimum-K estimation."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB).astype(np.float32)
    mean_lab = lab.reshape(-1, 3).mean(axis=0)
    return float(np.mean(np.linalg.norm(lab - mean_lab, axis=2)))

def recommend_min_k_for_image(image, candidates=MIN_K_CANDIDATES, threshold=RETENTION_THRESHOLD):
    """Find the smallest K whose color retention reaches the threshold."""
    spread = max(lab_color_spread_for_k(image), 1e-6)
    rows = []
    recommended = None
    for candidate_k in candidates:
        (candidate_img, candidate_palette), runtime = timed_call(kmeans_quantization, image, candidate_k, 2)
        error = quantization_error(image, candidate_img)
        retention = max(0.0, 1.0 - error / spread)
        rows.append({
            "K": candidate_k,
            "runtime_sec": runtime,
            "color_error": error,
            "retention": retention,
            "palette_size": len(candidate_palette),
        })
        if retention >= threshold and recommended is None:
            recommended = candidate_k
    return recommended if recommended is not None else candidates[-1], rows, spread


## 최종 파이프라인 실행

In [ ]:
def merge_shadow_label_fragments(label_map, palette, max_area=450, color_delta=45, passes=2):
    """Merge small low-contrast label islands into their surrounding color.

    Shadows and soft shading often become tiny K-Means regions. For each small
    connected component, merge it into the most common neighboring label only
    when the palette colors are close enough in Lab space.
    """
    labels = np.asarray(label_map, dtype=np.int32).copy()
    palette = np.asarray(palette, dtype=np.uint8)
    lab_palette = cv2.cvtColor(palette.reshape(1, -1, 3), cv2.COLOR_RGB2LAB).astype(np.float32)[0]
    kernel = np.ones((3, 3), np.uint8)

    for _ in range(passes):
        changed = False
        for label_id in np.unique(labels):
            mask = (labels == label_id).astype(np.uint8)
            n_labels, components, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
            for component_id in range(1, n_labels):
                area = int(stats[component_id, cv2.CC_STAT_AREA])
                if area > max_area:
                    continue
                component = components == component_id
                ring = cv2.dilate(component.astype(np.uint8), kernel, iterations=1).astype(bool) & ~component
                neighbor_labels = labels[ring]
                neighbor_labels = neighbor_labels[neighbor_labels != label_id]
                if neighbor_labels.size == 0:
                    continue
                values, counts = np.unique(neighbor_labels, return_counts=True)
                target_label = int(values[np.argmax(counts)])
                delta = float(np.linalg.norm(lab_palette[label_id] - lab_palette[target_label]))
                if delta <= color_delta:
                    labels[component] = target_label
                    changed = True
        if not changed:
            break
    return labels

def object_first_edges(label_map, min_area=260, close_kernel=5, thickness=1):
    """Create single-pass boundaries from cleaned color-object labels.

    Drawing each label's contour separately can place two parallel strokes on a
    shared boundary. This builds a cleaned label map first, then marks each
    label transition once so adjacent regions share one line.
    """
    labels = np.asarray(label_map, dtype=np.int32)
    cleaned_labels = np.full(labels.shape, -1, dtype=np.int32)
    kernel = np.ones((close_kernel, close_kernel), np.uint8)

    for label_id in np.unique(labels):
        mask = (labels == label_id).astype(np.uint8) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
        n_labels, components, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        for component_id in range(1, n_labels):
            if stats[component_id, cv2.CC_STAT_AREA] < min_area:
                continue
            cleaned_labels[components == component_id] = label_id

    padded = np.pad(cleaned_labels, 1, mode="constant", constant_values=-1)
    edge = np.zeros(labels.shape, dtype=np.uint8)
    center = padded[1:-1, 1:-1]
    foreground = center >= 0
    for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        neighbor = padded[1 + dy:1 + dy + labels.shape[0], 1 + dx:1 + dx + labels.shape[1]]
        edge[(foreground | (neighbor >= 0)) & (center != neighbor)] = 255

    if thickness > 1:
        edge = cv2.dilate(edge, np.ones((3, 3), np.uint8), iterations=thickness - 1)
    return edge

def thin_binary_edges(edges):
    """Thin binary edge strokes with Zhang-Suen thinning."""
    img = (edges > 0).astype(np.uint8)
    changed = True
    while changed:
        changed = False
        for step in (0, 1):
            padded = np.pad(img, 1, mode="constant")
            marker = np.zeros_like(img)
            for y in range(1, padded.shape[0] - 1):
                for x in range(1, padded.shape[1] - 1):
                    if padded[y, x] == 0:
                        continue
                    p2 = padded[y - 1, x]
                    p3 = padded[y - 1, x + 1]
                    p4 = padded[y, x + 1]
                    p5 = padded[y + 1, x + 1]
                    p6 = padded[y + 1, x]
                    p7 = padded[y + 1, x - 1]
                    p8 = padded[y, x - 1]
                    p9 = padded[y - 1, x - 1]
                    neighbors = [p2, p3, p4, p5, p6, p7, p8, p9]
                    count = sum(neighbors)
                    transitions = sum((neighbors[i] == 0 and neighbors[(i + 1) % 8] == 1) for i in range(8))
                    if not (2 <= count <= 6 and transitions == 1):
                        continue
                    if step == 0:
                        if p2 * p4 * p6 == 0 and p4 * p6 * p8 == 0:
                            marker[y - 1, x - 1] = 1
                    else:
                        if p2 * p4 * p8 == 0 and p2 * p6 * p8 == 0:
                            marker[y - 1, x - 1] = 1
            if np.any(marker):
                img[marker == 1] = 0
                changed = True
    return (img * 255).astype(np.uint8)

def detail_expression_edges(
    image,
    object_edges,
    low=100,
    high=220,
    min_area=8,
    max_area=450,
    dark_threshold=70,
    dark_min_area=5,
    dark_max_area=900,
    suppress_radius=2,
):
    """Extract fine expression/detail strokes without affecting segmentation regions."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    suppress_kernel = np.ones((2 * suppress_radius + 1, 2 * suppress_radius + 1), np.uint8)
    protected_edges = cv2.dilate(object_edges, suppress_kernel, iterations=1)

    # Thin edge details such as mouth curves and small wrinkles.
    detail = cv2.Canny(gray, low, high)
    detail = cv2.bitwise_and(detail, cv2.bitwise_not(protected_edges))
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(detail, 8)
    filtered_edges = np.zeros_like(detail)
    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if min_area <= area <= max_area:
            filtered_edges[labels == i] = 255

    # Dark filled details such as pupils, mouths, bats, and tiny facial marks.
    dark = (gray < dark_threshold).astype(np.uint8) * 255
    dark = cv2.bitwise_and(dark, cv2.bitwise_not(protected_edges))
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dark, 8)
    filtered_dark = np.zeros_like(dark)
    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if dark_min_area <= area <= dark_max_area:
            filtered_dark[labels == i] = 255

    return cv2.bitwise_or(filtered_edges, filtered_dark)

def border_connected_region_ids(region_map):
    """Return connected-component ids touching the image border; these are page background."""
    if region_map.size == 0:
        return set()
    border_ids = np.concatenate([
        region_map[0, :],
        region_map[-1, :],
        region_map[:, 0],
        region_map[:, -1],
    ])
    return set(int(v) for v in np.unique(border_ids) if int(v) > 0)

def best_label_point(region_map, region_id, bbox):
    x, y, w, h = bbox
    component = (region_map[y:y + h, x:x + w] == region_id).astype(np.uint8)
    if component.size == 0 or np.count_nonzero(component) == 0:
        return x + w / 2, y + h / 2
    dist = cv2.distanceTransform(component, cv2.DIST_L2, 5)
    _, _, _, max_loc = cv2.minMaxLoc(dist)
    return x + max_loc[0], y + max_loc[1]

def main_object_mask_from_regions(region_map, regions, background_ids, source_image=None, bridge_iterations=3):
    """Keep the main subject silhouette so isolated paper/background texture is ignored."""
    if source_image is not None:
        border = np.concatenate([
            source_image[0, :, :],
            source_image[-1, :, :],
            source_image[:, 0, :],
            source_image[:, -1, :],
        ], axis=0)
        bg = np.median(border.reshape(-1, 3), axis=0)
        diff = np.linalg.norm(source_image.astype(np.float32) - bg.astype(np.float32), axis=2)
        gray_src = cv2.cvtColor(source_image, cv2.COLOR_RGB2GRAY)
        seed = ((diff > 28) | (gray_src < 120)).astype(np.uint8) * 255
        seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
        seed = cv2.morphologyEx(seed, cv2.MORPH_CLOSE, np.ones((17, 17), np.uint8), iterations=2)
        seed = cv2.dilate(seed, np.ones((5, 5), np.uint8), iterations=2)

        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(seed, 8)
        if n_labels > 1:
            largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
            mask = (labels == largest).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            filled = np.zeros_like(mask)
            if contours:
                cv2.drawContours(filled, contours, -1, 255, -1)
                filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, np.ones((21, 21), np.uint8), iterations=1)
                return filled

    candidate = np.zeros(region_map.shape, dtype=np.uint8)
    for region in regions:
        region_id = int(region["id"])
        if region_id in background_ids:
            continue
        candidate[region_map == region_id] = 255

    if np.count_nonzero(candidate) == 0:
        return candidate

    bridge = cv2.dilate(candidate, np.ones((3, 3), np.uint8), iterations=bridge_iterations)
    bridge = cv2.morphologyEx(bridge, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8), iterations=1)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bridge, 8)
    if n_labels <= 1:
        return bridge

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    mask = (labels == largest).astype(np.uint8) * 255
    mask = cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=1)
    return mask

def draw_paint_by_number_style(
    line_image,
    detail_edges,
    regions,
    region_map,
    background_ids,
    source_image=None,
    line_gray=185,
    text_gray=155,
    font_scale=0.42,
):
    """Render white background, thin gray object lines, and small gray numbers only inside the main object."""
    if line_image.ndim == 3:
        gray = cv2.cvtColor(line_image, cv2.COLOR_RGB2GRAY)
    else:
        gray = line_image

    object_mask = main_object_mask_from_regions(region_map, regions, background_ids, source_image=source_image)
    keep_near_object = object_mask > 0
    edge_mask = ((gray < 128) | (detail_edges > 0)) & keep_near_object

    canvas = np.full((gray.shape[0], gray.shape[1], 3), 255, dtype=np.uint8)
    canvas[edge_mask] = line_gray

    font = cv2.FONT_HERSHEY_SIMPLEX
    for region in sorted(regions, key=lambda r: r["area"], reverse=True):
        region_id = int(region["id"])
        if region_id in background_ids:
            continue
        if region["area"] < MIN_REGION_AREA:
            continue
        region_pixels = region_map == region_id
        object_overlap = np.count_nonzero(region_pixels & keep_near_object)
        if object_overlap == 0:
            continue
        if object_overlap / max(1, int(region["area"])) < 0.45:
            continue
        text = str(region.get("color_id", region_id))
        cx, cy = best_label_point(region_map, region_id, region["bbox"])
        scale = font_scale
        thickness = 1
        (tw, th), base = cv2.getTextSize(text, font, scale, thickness)
        x0, y0, w, h = region["bbox"]
        x = int(np.clip(cx - tw / 2, x0 + 1, max(x0 + 1, x0 + w - tw - 1)))
        y = int(np.clip(cy + th / 2, y0 + th + 1, max(y0 + th + 1, y0 + h - 1)))
        cv2.putText(canvas, text, (x, y), font, scale, (text_gray, text_gray, text_gray), thickness, cv2.LINE_AA)
    return canvas


def extract_ink_line_edges(image, threshold=135, neutral_delta=85, min_area=8, dilate_iter=1):
    """Extract dark printed line art so whiskers and outlines stay as strokes."""
    arr = np.asarray(image, dtype=np.uint8)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    channel_delta = arr.max(axis=2).astype(np.int16) - arr.min(axis=2).astype(np.int16)
    dark_neutral = ((gray < threshold) & (channel_delta < neutral_delta)).astype(np.uint8) * 255
    dark_strong = (gray < max(65, threshold - 55)).astype(np.uint8) * 255
    ink = cv2.bitwise_or(dark_neutral, dark_strong)
    ink = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8), iterations=1)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(ink, 8)
    filtered = np.zeros_like(ink)
    for i in range(1, n_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            filtered[labels == i] = 255
    if dilate_iter > 0:
        filtered = cv2.dilate(filtered, np.ones((3, 3), np.uint8), iterations=dilate_iter)
    return filtered


def fill_masked_labels_from_neighbors(label_map, mask, max_iter=24):
    """Replace ink-line label pixels with neighboring color labels."""
    labels = np.asarray(label_map, dtype=np.int32).copy()
    unknown = mask > 0
    labels[unknown] = -1
    kernel = np.ones((3, 3), np.uint8)

    for _ in range(max_iter):
        remaining = labels < 0
        if not np.any(remaining):
            break
        changed = False
        for label_id in np.unique(labels[labels >= 0]):
            grow = cv2.dilate((labels == label_id).astype(np.uint8), kernel, iterations=1).astype(bool)
            targets = grow & remaining
            if np.any(targets):
                labels[targets] = int(label_id)
                changed = True
        if not changed:
            break

    if np.any(labels < 0):
        fallback = int(np.bincount(label_map.reshape(-1).astype(np.int32)).argmax())
        labels[labels < 0] = fallback
    return labels

def save_result_grid(path, items, cols=2, figsize=(10, 10)):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, (title, item_image) in zip(axes, items):
        ax.imshow(item_image, cmap="gray" if item_image.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def process_image(image_path):
    stem = Path(image_path).stem
    image = load_image(image_path)
    estimation_image = load_image(image_path, max_size=K_ESTIMATION_MAX_SIZE)
    min_recommended_k, _, _ = recommend_min_k_for_image(estimation_image)
    k = min_recommended_k + K_MARGIN

    start_total = time.perf_counter()
    simplified_image = cv2.bilateralFilter(
        image,
        SIMPLIFY_DIAMETER,
        SIMPLIFY_SIGMA_COLOR,
        SIMPLIFY_SIGMA_SPACE,
    )

    (kmeans_img, kmeans_palette, kmeans_labels), kmeans_time = timed_call(
        kmeans_quantization_with_labels,
        simplified_image,
        k,
    )
    if SHADOW_MERGE_ENABLED:
        kmeans_labels = merge_shadow_label_fragments(
            kmeans_labels,
            kmeans_palette,
            max_area=SHADOW_MERGE_MAX_AREA,
            color_delta=SHADOW_MERGE_COLOR_DELTA,
            passes=SHADOW_MERGE_PASSES,
        )
        kmeans_img = kmeans_palette[kmeans_labels]

    ink_edges = extract_ink_line_edges(
        image,
        threshold=INK_LINE_THRESHOLD,
        neutral_delta=INK_NEUTRAL_DELTA,
        min_area=INK_MIN_AREA,
        dilate_iter=0,
    )
    ink_label_mask = ink_edges
    if INK_MASK_DILATE > 0:
        ink_label_mask = cv2.dilate(ink_edges, np.ones((3, 3), np.uint8), iterations=INK_MASK_DILATE)
    kmeans_labels = fill_masked_labels_from_neighbors(kmeans_labels, ink_label_mask)
    kmeans_img = kmeans_palette[kmeans_labels]

    object_raw = object_first_edges(
        kmeans_labels,
        min_area=OBJECT_MIN_AREA,
        close_kernel=OBJECT_CLOSE_KERNEL,
        thickness=LINE_THICKNESS,
    )
    object_seg_clean = clean_edges(object_raw, open_iter=0, close_iter=1, thickness=2)
    object_final_clean = thin_binary_edges(object_seg_clean)
    detail_edges = detail_expression_edges(
        image,
        object_final_clean,
        low=DETAIL_CANNY_LOW,
        high=DETAIL_CANNY_HIGH,
        min_area=DETAIL_MIN_AREA,
        max_area=DETAIL_MAX_AREA,
        dark_threshold=DARK_DETAIL_THRESHOLD,
        dark_min_area=DARK_DETAIL_MIN_AREA,
        dark_max_area=DARK_DETAIL_MAX_AREA,
        suppress_radius=DETAIL_SUPPRESS_RADIUS,
    )
    detail_edges = cv2.bitwise_or(detail_edges, ink_edges)

    segmentation_line_image = coloring_line_image(object_seg_clean)
    final_line_image = coloring_line_image(object_final_clean)
    (region_map, regions), seg_time = timed_call(segment_connected_components, segmentation_line_image, MIN_REGION_AREA)
    background_color = estimate_background_color(kmeans_img)
    regions = assign_region_color_numbers(
        regions,
        region_map,
        kmeans_labels,
        kmeans_palette,
        background_color=background_color,
        background_color_threshold=16,
        merge_background_similar=True,
    )
    background_region_ids = border_connected_region_ids(region_map)
    for region in regions:
        if int(region["id"]) in background_region_ids:
            region["is_background"] = True

    colored_by_labels = np.full_like(simplified_image, 255)
    for region in regions:
        color = region.get("color_rgb")
        if color is None:
            continue
        colored_by_labels[region_map == region["id"]] = np.array(color, dtype=np.uint8)
    colored_by_labels[segmentation_line_image < 128] = 0
    colored_by_labels[detail_edges > 0] = 0

    numbered_coloringbook = draw_paint_by_number_style(
        final_line_image,
        detail_edges,
        regions,
        region_map,
        background_region_ids,
        source_image=simplified_image,
    )
    colored_by_labels_numbered = label_regions(
        colored_by_labels,
        regions,
        font_scale=0.65,
        region_map=region_map,
        skip_background=True,
    )
    color_index = color_index_table_image(kmeans_palette, regions, "Color Index")

    input_path = f"{OUTPUT_DIR}/{stem}_input.png"
    grid_path = f"{OUTPUT_DIR}/{stem}_grid.png"
    save_image_rgb(input_path, image)
    save_result_grid(
        grid_path,
        [
            ("Input", image),
            (f"Final Coloring Book (K={k})", numbered_coloringbook),
            ("Colored Preview", colored_by_labels_numbered),
            ("Color Index", color_index),
        ],
        cols=2,
        figsize=(11, 10),
    )

    return {
        "image": image_path,
        "input_path": input_path,
        "grid_path": grid_path,
        "min_recommended_k": min_recommended_k,
        "K": k,
        "regions": len([r for r in regions if not r.get("is_background", False)]),
        "kmeans_time": kmeans_time,
        "seg_time": seg_time,
        "total_time": time.perf_counter() - start_total,
    }


batch_rows = []
for image_path in IMAGE_PATHS:
    print(f"Processing {image_path}...")
    batch_rows.append(process_image(image_path))

print()
print("Saved only input/grid files to:", OUTPUT_DIR)
print_table(batch_rows, columns=["image", "min_recommended_k", "K", "regions", "input_path", "grid_path", "total_time"])


## RGB 색상표

In [ ]:
# Palette images are no longer saved separately.
# Each image's color index is included inside outputs/<image>_grid.png.


## 정량 평가 표

In [ ]:
# Batch summary is printed by the main pipeline cell above.


## 색상 개수별 복잡도 변화

In [ ]:
# K complexity comparison is disabled for batch output mode.
# The pipeline now saves only outputs/<image>_input.png and outputs/<image>_grid.png.


## 자동 정리: 알고리즘 장단점

### 색상 단순화

- K-Means 장점: 이미지 색상 분포를 반영해 대표색을 찾으므로 색상 유지 품질이 좋습니다.
- K-Means 단점: 반복 계산이 필요해 Posterization보다 느립니다.
- Posterization 장점: 매우 빠르고 구현이 단순합니다.
- Posterization 단점: 실제 이미지 색상 분포와 무관하게 균등 분할하므로 부자연스러운 색상 계단이 생길 수 있습니다.
- Median Cut 장점: 색상 분포 범위를 재귀적으로 나누므로 교육용 설명과 비교 실험에 좋습니다.
- Median Cut 단점: 고품질 최적화는 K-Means보다 약하고 구현 방식에 따라 속도 차이가 큽니다.
- 정량 비교: `color_error`는 원본과 양자화 이미지의 Lab 색 거리 평균이고 낮을수록 원본 색 보존이 좋습니다. `unique_colors`는 결과 이미지에 실제로 남은 고유색 수입니다.

### 경계선 추출

- Sobel 장점: 방향별 밝기 변화 설명이 쉽고 빠릅니다.
- Sobel 단점: 선이 두껍거나 질감 변화에 민감할 수 있습니다.
- Laplacian 장점: 모든 방향의 급격한 변화를 한 번에 감지합니다.
- Laplacian 단점: 노이즈와 작은 텍스처까지 경계로 잡는 경향이 있습니다.
- Canny 장점: 노이즈 제거와 이중 임계값을 포함해 선명하고 안정적인 경계를 제공합니다.
- Canny 단점: 밝기가 비슷한 서로 다른 색의 경계는 놓칠 수 있습니다.
- Hybrid Color Boundary 장점: Canny 결과에 색상 라벨 경계를 합쳐 겹친 색상 영역도 분리합니다.
- Hybrid Color Boundary 단점: K-Means 결과가 과분할되면 선이 많아져 복잡도가 올라갈 수 있습니다.

### 영역 분리

- Connected Components 장점: 닫힌 흰 영역을 직접 분리하고, 각 영역에 dominant 색상 번호를 넣기 쉬워 최종 산출물에 적합합니다.
- Connected Components 단점: 선이 끊기면 영역이 합쳐질 수 있어 Morphology 보정이 중요합니다.
- 배경 번호 병합 장점: 외곽에서 추정한 배경색과 비슷한 내부 빈 공간도 같은 배경 번호로 표시되어 색상표와 일관됩니다.
- Contour Detection 장점: 외곽선 시각화와 영역 형태 확인이 쉽습니다.
- Contour Detection 단점: 내부 계층 구조 처리가 필요할 수 있습니다.
- Watershed 장점: 겹친 객체 분리에 강합니다.
- Watershed 단점: 컬러링북에서는 과분할로 인해 사용자가 색칠하기 어려운 작은 영역이 늘 수 있습니다.

## 최종 알고리즘 선택 이유

최종 조합은 `K-Means + Hybrid Canny/Color Boundary + Connected Components`입니다. K-Means는 원본 색상 보존과 색상 수 제어의 균형이 좋고, Hybrid 경계선은 밝기가 비슷한 색상 사이의 경계까지 보완하며, Connected Components는 색상 번호를 넣을 색칠 영역을 직접 계산하기 좋습니다. Contour Detection과 Watershed는 최종 선택이 아니라 비교와 설명용 보조 결과로 저장합니다.

## Trade-off 분석

- K 증가: 원본과 비슷해지지만 영역 수와 번호 수가 늘어 색칠 난이도가 올라갑니다.
- 선 두께 증가: 경계 가독성은 좋아지지만 색칠 공간이 줄고 작은 영역이 사라질 수 있습니다.
- 최소 영역 크기 증가: 번호 가독성은 좋아지지만 세부 묘사가 줄어듭니다.
- Canny 임계값 증가: 노이즈는 줄지만 필요한 경계가 누락될 수 있습니다.

## 발표용 비교 포인트

- 같은 이미지에서 K=5, 10, 20, 50 결과를 비교하며 복잡도 변화를 설명합니다.
- K-Means, Posterization, Median Cut의 색상 오차와 실제 고유색 수를 비교합니다.
- Sobel, Laplacian, Canny, Hybrid의 Edge Density와 시각적 노이즈를 비교합니다.
- 영역 개수, 평균 영역 크기, 작은 영역 개수를 HCI 지표로 연결합니다.
- 최종 결과는 K-Means 색상표 번호와 영역 번호가 연결되므로, 같은 색상 영역은 같은 숫자로 칠할 수 있습니다.

## HCI 관점 개선점

- 색상 단순화로 사용자가 선택해야 할 색의 수를 줄였습니다.
- 검은 선/흰 배경 형태로 출력해 인쇄와 색칠에 적합하게 만들었습니다.
- 작은 영역 제거로 색칠 스트레스와 번호 혼잡을 줄였습니다.
- 중심 기반 번호 배치와 겹침 회피로 번호 인식 편의성을 높였습니다.
- 성능 표와 복잡도 그래프로 사용자 난이도와 알고리즘 파라미터의 관계를 설명할 수 있습니다.